Usaremos novamente o conjunto de dados aberto do popular site <a href="https://www.kaggle.com">Kaggle</a>, que usamos na Semana 1 para o nosso exemplo.

Lembre-se de que este <a href="https://www.kaggle.com/hugomathien/soccer">Banco de Dados de Futebol Europeu</a> possui mais de 25.000 partidas e mais de 10.000 jogadores das temporadas profissionais de futebol europeu de 2008 a 2016.

**Observação:** Baixe o arquivo *database.sqlite* caso ainda não o tenha na sua pasta *Week-7-MachineLearning*.

Imports

In [ ]:
import sqlite3
import pandas as pd 
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Leitura dos dados da base de dados com Pandas.
<br><br></p>

In [5]:
cnx = sqlite3.connect(r'C:\Users\mathe\Downloads\archive (5)\database.sqlite')
df = pd.read_sql_query("SELECT * FROM Player_Attributes", cnx)

In [6]:
df.head(5)

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0


In [7]:
df.shape

(183978, 42)

In [8]:
df.columns

Index(['id', 'player_fifa_api_id', 'player_api_id', 'date', 'overall_rating',
       'potential', 'preferred_foot', 'attacking_work_rate',
       'defensive_work_rate', 'crossing', 'finishing', 'heading_accuracy',
       'short_passing', 'volleys', 'dribbling', 'curve', 'free_kick_accuracy',
       'long_passing', 'ball_control', 'acceleration', 'sprint_speed',
       'agility', 'reactions', 'balance', 'shot_power', 'jumping', 'stamina',
       'strength', 'long_shots', 'aggression', 'interceptions', 'positioning',
       'vision', 'penalties', 'marking', 'standing_tackle', 'sliding_tackle',
       'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning',
       'gk_reflexes'],
      dtype='object')

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Declarando as colunas que queremos usar na feature.
<br><br></p>

In [9]:
feature = [
       'potential', 'crossing', 'finishing', 'heading_accuracy',
       'short_passing', 'volleys', 'dribbling', 'curve', 'free_kick_accuracy',
       'long_passing', 'ball_control', 'acceleration', 'sprint_speed',
       'agility', 'reactions', 'balance', 'shot_power', 'jumping', 'stamina',
       'strength', 'long_shots', 'aggression', 'interceptions', 'positioning',
       'vision', 'penalties', 'marking', 'standing_tackle', 'sliding_tackle',
       'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning',
       'gk_reflexes']

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Especificando o alvo da predição
<br><br></p>

In [13]:
target = ['overall_rating']

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Limpando os dados.
<br><br></p>

In [14]:
df = df.dropna() # Retiando valores nulos

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Especifique o Alvo da Previsão
<br><br></p>

In [ ]:
x = df[feature] # Criando um novo dataframe que contém apenas as colunas de feature

In [ ]:
y = df[target] # Criando um novo dataframe com contém apenas as colunas de target 'overrall_rating'

Vajamos uma linha típica de nossos recursos (feature)

In [ ]:
x.iloc[2] # Acessando índice 2 do dataframe x

potential             66.0
crossing              49.0
finishing             44.0
heading_accuracy      71.0
short_passing         61.0
volleys               44.0
dribbling             51.0
curve                 45.0
free_kick_accuracy    39.0
long_passing          64.0
ball_control          49.0
acceleration          60.0
sprint_speed          64.0
agility               59.0
reactions             47.0
balance               65.0
shot_power            55.0
jumping               58.0
stamina               54.0
strength              76.0
long_shots            35.0
aggression            63.0
interceptions         41.0
positioning           45.0
vision                54.0
penalties             48.0
marking               65.0
standing_tackle       66.0
sliding_tackle        69.0
gk_diving              6.0
gk_handling           11.0
gk_kicking            10.0
gk_positioning         8.0
gk_reflexes            8.0
Name: 2, dtype: float64

In [18]:
y

,overall_rating
0,67.0
1,67.0
2,62.0
3,61.0
4,61.0
...,...
183973,83.0
183974,78.0
183975,77.0
183976,78.0


<p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Dividindo o dataset em treino e dataset test.</p>

In [19]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.33, random_state=324 )

<p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Regressão linear: Ajustar um modelo ao conjunto de treinamento.</p>

In [20]:
regressor = LinearRegression() # Cria um modelo de regressão Linear
regressor.fit(x_train, y_train) # Treina o modelo com entrada com e saída

LinearRegression()

<p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Realizar previsão usando modelo de regressão linear</p>

In [21]:
y_prediction = regressor.predict(x_test)
y_prediction

array([[66.51284879],
       [79.77234615],
       [66.57371825],
       ...,
       [69.23780133],
       [64.58351696],
       [73.6881185 ]])

<p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Qual é a média do valor alvo esperado no conjunto de teste?</p>

In [22]:
y_test.describe()

,overall_rating
count,59517.000000
mean,68.635818
std,7.041297
min,33.000000
25%,64.000000
50%,69.000000
75%,73.000000
max,94.000000


<p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Avalie a precisão da regressão linear usando a raiz do erro quadrático médio</p>

In [23]:
#Root Mean Squared Error (RMSE)
#Pega a diferença entre o que o modelo previu e o que era o valor real, eleva ao quadrado, soma tudo e tira a média.
RMSE = sqrt(mean_squared_error(y_true= y_test, y_pred= y_prediction)) 

In [24]:
print(RMSE)

2.8053030468552143



 <p style="font-family: Arial; font-size:1.75em;color:#2462C0; font-style:bold">
Regressor de árvore de decisão: ajuste um novo modelo de regressão ao conjunto de treinamento</p>

In [25]:
regressor = DecisionTreeRegressor(max_depth=20)
regressor.fit(x_train, y_train)

DecisionTreeRegressor(max_depth=20)

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Executar previsão usando o regressor de árvore de decisão
<br><br></p>

In [26]:
y_prediction = regressor.predict(x_test)
y_prediction

array([62.        , 84.        , 62.38666667, ..., 71.        ,
       62.        , 73.        ])

<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Para comparação: Qual é a média do valor alvo esperado no conjunto de teste?
<br><br></p>

In [27]:
y_test.describe()

,overall_rating
count,59517.000000
mean,68.635818
std,7.041297
min,33.000000
25%,64.000000
50%,69.000000
75%,73.000000
max,94.000000


<p style="font-family: Arial; font-size:1.75em;color:purple; font-style:bold"><br>

Avalie a precisão da regressão da árvore de decisão usando a raiz do erro quadrático médio

<br><br></p>

In [28]:
RMSE = sqrt(mean_squared_error(y_true = y_test, y_pred = y_prediction))
print(RMSE)

1.4529681858326677
